# 14 Combining and reshaping

Joining tables on a key, stacking tables, and moving between long and wide shapes.

In [1]:
import pandas as pd

patients = pd.read_csv("../data/patients.csv")
visits = pd.read_csv("../data/visits.csv", parse_dates=["visit_date"])
print(patients.shape, visits.shape)
visits.head()

(300, 8) (727, 6)


,patient_id,visit,visit_date,hb,wbc,crp
0,P0001,0,2025-09-18,12.5,5.5,2.6
1,P0001,1,2025-10-16,13.6,5.9,2.5
2,P0002,0,2025-03-25,13.9,13.9,1.6
3,P0002,1,2025-04-17,14.4,9.0,3.3
4,P0003,0,2025-03-25,15.0,7.2,10.9


## merge

`how` decides which rows are kept:

| how | keeps |
|---|---|
| `inner` | keys in both tables |
| `left` | every row of the left table |
| `right` | every row of the right table |
| `outer` | every key from either |

In [2]:
merged = visits.merge(patients[["patient_id", "site", "sex", "age"]], on="patient_id", how="left")
merged.head()

,patient_id,visit,visit_date,hb,wbc,crp,site,sex,age
0,P0001,0,2025-09-18,12.5,5.5,2.6,Kampala,M,30
1,P0001,1,2025-10-16,13.6,5.9,2.5,Kampala,M,30
2,P0002,0,2025-03-25,13.9,13.9,1.6,Mbarara,M,81
3,P0002,1,2025-04-17,14.4,9.0,3.3,Mbarara,M,81
4,P0003,0,2025-03-25,15.0,7.2,10.9,Gulu,F,24


`validate` checks the relationship you expect, and `indicator` shows where each row came from.
Use them: a wrong join is one of the most common silent errors in analysis.

In [3]:
check = patients.merge(visits, on="patient_id", how="outer",
                       validate="one_to_many", indicator=True)
check["_merge"].value_counts()

_merge
both          727
left_only       0
right_only      0
Name: count, dtype: int64

Different key names in each table:

In [4]:
codes = pd.DataFrame({"site_name": ["Kampala", "Gulu", "Mbarara", "Mbale", "Jinja"],
                      "region": ["Central", "Northern", "Western", "Eastern", "Eastern"]})
patients.merge(codes, left_on="site", right_on="site_name", how="left")[["patient_id", "site", "region"]].head()

,patient_id,site,region
0,P0001,Kampala,Central
1,P0002,Mbarara,Western
2,P0003,Gulu,Northern
3,P0004,Gulu,Northern
4,P0005,Gulu,Northern


## concat: stacking

Same columns, more rows (or `axis=1` for side by side).

In [5]:
batch1 = patients.iloc[:3]
batch2 = patients.iloc[3:5]
pd.concat([batch1, batch2], ignore_index=True)

,patient_id,site,sex,age,weight_kg,height_cm,hiv_status,enrolled
0,P0001,Kampala,M,30,58.1,150.4,Negative,2025-09-20
1,P0002,Mbarara,M,81,61.4,164.4,Negative,2025-03-23
2,P0003,Gulu,F,24,63.5,169.5,Negative,2025-03-22
3,P0004,Gulu,F,8,NaN,153.5,Positive,2025-04-17
4,P0005,Gulu,M,74,73.3,159.3,Negative,2025-03-24


In [6]:
pd.concat([batch1.assign(batch=1), batch2.assign(batch=2)])[["patient_id", "batch"]]

,patient_id,batch
0,P0001,1
1,P0002,1
2,P0003,1
3,P0004,2
4,P0005,2


## Long and wide

- long: one row per measurement (patient, visit, value). Best for groupby and seaborn.
- wide: one row per patient, a column per visit. Best for some tables and for ML features.

In [7]:
wide = visits.pivot(index="patient_id", columns="visit", values="hb")
wide.columns = [f"hb_v{c}" for c in wide.columns]
wide.head()

,hb_v0,hb_v1,hb_v2,hb_v3
patient_id,,,,
P0001,12.5,13.6,NaN,NaN
P0002,13.9,14.4,NaN,NaN
P0003,15.0,14.6,NaN,NaN
P0004,11.5,13.0,NaN,NaN
P0005,11.4,11.6,NaN,NaN


`pivot_table` also summarises when a combination appears more than once:

In [8]:
merged.pivot_table(index="site", columns="sex", values="hb", aggfunc="mean").round(2)

sex,F,M
site,,
Gulu,13.31,12.46
Jinja,12.32,12.62
Kampala,12.74,12.37
Mbale,12.92,12.77
Mbarara,12.15,13.04


## melt: wide back to long

In [9]:
long_again = wide.reset_index().melt(id_vars="patient_id", var_name="visit", value_name="hb")
long_again = long_again.dropna()
long_again["visit"] = long_again["visit"].str.replace("hb_v", "").astype(int)
long_again.sort_values(["patient_id", "visit"]).head()

,patient_id,visit,hb
0,P0001,0,12.5
300,P0001,1,13.6
1,P0002,0,13.9
301,P0002,1,14.4
2,P0003,0,15.0


Several measures at once:

In [10]:
labs_long = visits.melt(id_vars=["patient_id", "visit"], value_vars=["hb", "wbc", "crp"],
                        var_name="test", value_name="value")
labs_long.head()

,patient_id,visit,test,value
0,P0001,0,hb,12.5
1,P0001,1,hb,13.6
2,P0002,0,hb,13.9
3,P0002,1,hb,14.4
4,P0003,0,hb,15.0


## Change from baseline

A common clinical calculation that combines grouping and shifting.

In [11]:
visits = visits.sort_values(["patient_id", "visit"])
visits["hb_baseline"] = visits.groupby("patient_id")["hb"].transform("first")
visits["hb_change"] = (visits["hb"] - visits["hb_baseline"]).round(1)
visits["hb_previous"] = visits.groupby("patient_id")["hb"].shift(1)
visits.head(8)

,patient_id,visit,visit_date,hb,wbc,crp,hb_baseline,hb_change,hb_previous
0,P0001,0,2025-09-18,12.5,5.5,2.6,12.5,0.0,NaN
1,P0001,1,2025-10-16,13.6,5.9,2.5,12.5,1.1,12.5
2,P0002,0,2025-03-25,13.9,13.9,1.6,13.9,0.0,NaN
3,P0002,1,2025-04-17,14.4,9.0,3.3,13.9,0.5,13.9
4,P0003,0,2025-03-25,15.0,7.2,10.9,15.0,0.0,NaN
5,P0003,1,2025-04-18,14.6,7.6,2.0,15.0,-0.4,15.0
6,P0004,0,2025-04-14,11.5,5.7,1.3,11.5,0.0,NaN
7,P0004,1,2025-05-16,13.0,15.6,1.9,11.5,1.5,11.5
